# 03 – Full load from an external API into the lake (`replace`)

**Goal (Task 2, part 4):** load data from a **public API related to our domain** into the lake.

**API:** [Dog CEO](https://dog.ceo/dog-api/) – free, no API key. Endpoint `https://dog.ceo/api/breeds/list/all` returns every dog breed and its sub-breeds.

**Why it fits PawHome:** our `animals` table stores dog breeds. A reference list of breeds in the lake lets an analyst check or enrich the breeds entered in the shelters.

The response looks like this:
```json
{"message": {"affenpinscher": [], "australian": ["kelpie", "shepherd"], ...}, "status": "success"}
```
We turn it into one row per breed / sub-breed. Keys: `dogceo/breeds/ingestion_date=YYYY-MM-DD/<load_id>.<file_id>.parquet`

In [ ]:
import os
import dlt
from dlt.sources.helpers import requests      # requests with automatic retries
from lake import PIPELINES_DIR, lake_destination, add_metadata, list_lake, read_lake

DOG_API_URL = os.getenv("DOG_API_URL", "https://dog.ceo/api/breeds/list/all")

## Step 1 – A dlt resource that calls the API

A resource is a Python function that **yields rows**. dlt takes care of the Parquet files and the upload.

In [ ]:
@dlt.resource(name="breeds", write_disposition="replace")
def dog_breeds():
    response = requests.get(DOG_API_URL, timeout=30)
    response.raise_for_status()                 # stop if the API answers with an error
    for breed, sub_breeds in response.json()["message"].items():
        if not sub_breeds:                      # breed without sub-breeds, e.g. "akita"
            yield {"breed": breed, "sub_breed": None, "full_name": breed,
                   "images_api": f"https://dog.ceo/api/breed/{breed}/images"}
        for sub in sub_breeds:                  # one row per sub-breed, e.g. "australian shepherd"
            yield {"breed": breed, "sub_breed": sub, "full_name": f"{breed} {sub}",
                   "images_api": f"https://dog.ceo/api/breed/{breed}/{sub}/images"}


def build_api_resource():
    return add_metadata(dog_breeds(), source_system="dogceo",
                        source_resource="breeds/list/all", ingestion_type="full")

## Step 2 – Pipeline and first run

In [ ]:
api_pipeline = dlt.pipeline(
    pipeline_name="dogceo_full",
    destination=lake_destination("silo"),
    dataset_name="dogceo",
    pipelines_dir=PIPELINES_DIR,
)
api_pipeline.run(build_api_resource(), loader_file_format="parquet")
print("Rows:", api_pipeline.last_trace.last_normalize_info.row_counts)
api_keys = list_lake("dogceo/breeds")
api_keys

In [ ]:
breeds = read_lake("dogceo/breeds/")
print(len(breeds), "rows")
breeds.head(10)

## Step 3 – Run again and compare

`replace` → the old file is removed, a new one with a new `load_id` is written. The row count only changes if the API's data changed.

In [ ]:
before = set(api_keys.key)
api_pipeline.run(build_api_resource(), loader_file_format="parquet")
after = set(list_lake("dogceo/breeds").key)
print("REMOVED:", sorted(before - after))
print("ADDED:  ", sorted(after - before))
print("Rows now:", len(read_lake("dogceo/breeds/")))

**Bonus idea:** compare the lake's breeds with the breeds used in our shelters (both now live in the lake):

In [ ]:
shelter_dogs = read_lake("postgres/animals/")
shelter_dogs = shelter_dogs[shelter_dogs.species == "dog"]
known = set(breeds.breed.str.lower())
shelter_dogs["breed_in_api"] = shelter_dogs.breed.str.lower().str.split().str[-1].isin(known)
print(f"{shelter_dogs.breed_in_api.mean():.0%} of shelter dogs have a breed word that Dog CEO knows")